In [2]:
!pip install simpy mesa scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mesa]


In [3]:
import random
import simpy

# --- Settings (change these to experiment) ---
RANDOM_SEED = 42        # fixed seed so the results are repeatable
NUM_PACKETS = 100       # how many packets to send through the chain
ARRIVAL_MEAN = 1.5      # average time between new packets
ROUTER_DELAYS = [0.8, 1.2, 0.8]  # processing delay at Router A, B, C


def packet(env, packet_id, routers, results):
    """One packet travelling through the whole chain of routers."""
    arrival_time = env.now  # the clock time this packet was created

    # Visit each router in order. A router is a Resource with capacity 1,
    # so if it is busy, the packet waits in line (that waiting time is
    # part of the packet's total latency).
    for router, delay in zip(routers, ROUTER_DELAYS):
        with router.request() as request:
            yield request             # wait until the router is free
            yield env.timeout(delay)  # the router processes the packet

    # The packet has left the last router: record its total latency.
    results.append(env.now - arrival_time)


def packet_source(env, routers, results):
    """Creates packets one at a time, with random gaps between them."""
    for i in range(NUM_PACKETS):
        env.process(packet(env, i, routers, results))
        # Random gap until the next packet (exponential distribution,
        # which is the standard way to model random network arrivals).
        yield env.timeout(random.expovariate(1.0 / ARRIVAL_MEAN))


def main():
    random.seed(RANDOM_SEED)
    env = simpy.Environment()

    # Three routers in a chain. Capacity 1 = one packet at a time.
    routers = [simpy.Resource(env, capacity=1) for _ in range(3)]

    results = []  # total latency of every packet
    env.process(packet_source(env, routers, results))
    env.run()  # run until all packets have finished

    # --- Results ---
    average = sum(results) / len(results)
    minimum_possible = sum(ROUTER_DELAYS)  # latency with zero waiting

    print(f"Packets delivered: {len(results)}")
    print(f"Minimum possible latency (no waiting): {minimum_possible:.2f} time units")
    print(f"Average total latency: {average:.2f} time units")
    print(f"Fastest packet: {min(results):.2f} time units")
    print(f"Slowest packet: {max(results):.2f} time units")
    print(f"Average time spent waiting in line: {average - minimum_possible:.2f} time units")
    print()
    print("Why the average is higher than the minimum: Router B is the")
    print("slowest router, so packets sometimes queue up behind it. That")
    print("waiting time is what pushes total latency above the sum of the")
    print("processing delays.")


if __name__ == "__main__":
    main()


Packets delivered: 100
Minimum possible latency (no waiting): 2.80 time units
Average total latency: 5.39 time units
Fastest packet: 2.80 time units
Slowest packet: 11.35 time units
Average time spent waiting in line: 2.59 time units

Why the average is higher than the minimum: Router B is the
slowest router, so packets sometimes queue up behind it. That
waiting time is what pushes total latency above the sum of the
processing delays.


In [4]:
from mesa import Agent, Model
from mesa.datacollection import DataCollector


class RouterAgent(Agent):
    """A router that carries traffic for the users connected to it."""

    def __init__(self, model, router_id, capacity):
        super().__init__(model)
        self.router_id = router_id
        self.capacity = capacity  # max users it can handle without congestion
        self.load = 0             # users currently routed through it

    def reset(self):
        """Called at the start of every step: last step's load is gone."""
        self.load = 0

    def is_congested(self):
        return self.load >= self.capacity


class UserAgent(Agent):
    """A user sending traffic through one router at a time."""

    def __init__(self, model, user_id, router):
        super().__init__(model)
        self.user_id = user_id
        self.router = router      # the router this user is using right now
        self.reroutes = 0         # how many times this user has switched

    def step(self):
        # If my router is full, move to the least-loaded router instead.
        if self.router.is_congested():
            least_loaded = min(self.model.routers, key=lambda r: r.load)
            if least_loaded is not self.router:
                self.router = least_loaded
                self.reroutes += 1
        # Send my traffic through (whichever) router I am on now.
        self.router.load += 1


class NetworkModel(Model):
    """The whole network: 3 routers and a group of users."""

    def __init__(self, num_users=10, seed=42):
        super().__init__(seed=seed)
        # Three routers, each able to handle 4 users at once.
        self.routers = [RouterAgent(self, i, capacity=4) for i in range(3)]
        # Everyone starts on Router 0 (the default route). That is what
        # creates the congestion the model then fixes by rerouting.
        self.users = [UserAgent(self, i, self.routers[0]) for i in range(num_users)]
        self.step_reroutes = 0

        self.datacollector = DataCollector(
            model_reporters={
                "Reroutes": lambda m: m.step_reroutes,
                "Max load": lambda m: max(r.load for r in m.routers),
                "Congested routers": lambda m: sum(
                    1 for r in m.routers if r.is_congested()
                ),
            }
        )

    def step(self):
        # Phase 1: every router starts the step empty.
        for router in self.routers:
            router.reset()
        # Phase 2: users act in a random order (so no user always goes
        # first), each one picking a router and adding their traffic.
        before = sum(u.reroutes for u in self.users)
        order = self.users[:]
        self.random.shuffle(order)
        for user in order:
            user.step()
        self.step_reroutes = sum(u.reroutes for u in self.users) - before
        self.datacollector.collect(self)


def main():
    model = NetworkModel(num_users=10, seed=42)
    for _ in range(20):
        model.step()

    results = model.datacollector.get_model_vars_dataframe()
    print("Step-by-step results (first 10 steps):")
    print(results.head(10))
    print()
    print(f"Total reroutes over 20 steps: {sum(u.reroutes for u in model.users)}")
    print("Final users per router:", [r.load for r in model.routers])
    print()
    print("What happened: everyone started on Router 0, so it congested")
    print("immediately. Users moved to the emptier routers until the load")
    print("spread out across all three — that is adaptive rerouting.")


if __name__ == "__main__":
    main()


Step-by-step results (first 10 steps):
   Reroutes  Max load  Congested routers
0         6         4                  1
1         0         4                  1
2         0         4                  1
3         0         4                  1
4         0         4                  1
5         0         4                  1
6         0         4                  1
7         0         4                  1
8         0         4                  1
9         0         4                  1

Total reroutes over 20 steps: 6
Final users per router: [4, 3, 3]

What happened: everyone started on Router 0, so it congested
immediately. Users moved to the emptier routers until the load
spread out across all three — that is adaptive rerouting.


/usr/local/lib/python3.13/dist-packages/mesa/mesa_logging.py:112: FutureWarning: The use of the `seed` keyword argument is deprecated, use `rng` instead. No functional changes.
  res = func(*args, **kwargs)


In [5]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error


def make_traffic_data(n_samples=500, seed=42):
    """Simulate traffic measurements and the latency that resulted.

    Features (what the network looked like):
      - traffic: packets per second on the link
      - load:    how full the router was, in percent (0-100)
      - hour:    hour of the day (0-23), since traffic has daily patterns
    Target (what we want to predict):
      - latency: milliseconds for a packet to get through
    The latency formula below is the 'ground truth' of our simulated
    network: latency grows with traffic, and grows FASTER as the router
    fills up (that squared term is the queueing effect — a nearly-full
    router delays packets much more than a half-full one).
    """
    rng = np.random.default_rng(seed)
    traffic = rng.integers(100, 1000, size=n_samples)   # packets/sec
    load = rng.uniform(10, 95, size=n_samples)          # router load %
    hour = rng.integers(0, 24, size=n_samples)          # hour of day
    noise = rng.normal(0, 3, size=n_samples)            # random variation

    latency = (10 + 0.05 * traffic + 0.4 * load
               + 0.01 * load ** 2 + 2 * np.sin(hour / 24 * 2 * np.pi)
               + noise)
    X = np.column_stack([traffic, load, hour])
    return X, latency


def main():
    # ---------- Stage 1: train the model ----------
    X, y = make_traffic_data()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    print(f"Test samples: {len(y_test)}")
    print(f"Mean absolute error: {mae:.2f} ms "
          f"(on average, predictions are off by this much)")
    print(f"Model score (R^2): {model.score(X_test, y_test):.3f} "
          f"(1.0 would be perfect)")
    print()

    # ---------- Stage 2: use predictions to choose routes ----------
    # New traffic scenarios the model has never seen. For each one we
    # compare two options:
    #   Route A (current): keep the traffic where it is.
    #   Route B (alternate): a longer path (+15 ms detour), but it only
    #     carries 30% of the current load, so it is much less congested.
    print("Routing decisions for 10 new traffic scenarios:")
    print("traffic  load | predicted A | predicted B | decision")
    rng = np.random.default_rng(7)
    saved = []
    rerouted = 0
    for _ in range(10):
        traffic = int(rng.integers(100, 1000))
        load = float(rng.uniform(10, 95))
        hour = int(rng.integers(0, 24))

        pred_a = model.predict([[traffic, load, hour]])[0]
        pred_b = model.predict([[traffic, load * 0.3, hour]])[0] + 15

        if pred_b < pred_a:
            decision = "reroute to B"
            rerouted += 1
            saved.append(pred_a - pred_b)
        else:
            decision = "stay on A"
        print(f"{traffic:7d} {load:5.0f}% | {pred_a:10.1f} ms | {pred_b:10.1f} ms | {decision}")

    print()
    print(f"Rerouted {rerouted} of 10 scenarios.")
    if saved:
        print(f"Average predicted time saved by rerouting: {np.mean(saved):.1f} ms")
    print()
    print("What this shows: the model learned that latency explodes when")
    print("the router is nearly full, so at high load the alternate route")
    print("wins even with its 15 ms detour — and at low load, staying put")
    print("wins. The prediction, not a fixed rule, makes the call.")


if __name__ == "__main__":
    main()


Test samples: 100
Mean absolute error: 3.39 ms (on average, predictions are off by this much)
Model score (R^2): 0.989 (1.0 would be perfect)

Routing decisions for 10 new traffic scenarios:
traffic  load | predicted A | predicted B | decision
    950    86% |      165.9 ms |       83.6 ms | reroute to B
    620    29% |       59.4 ms |       66.7 ms | stay on A
    149    84% |      123.7 ms |       53.1 ms | reroute to B
    921    80% |      148.3 ms |       82.2 ms | reroute to B
    218    50% |       68.0 ms |       44.2 ms | reroute to B
    834    34% |       74.7 ms |       75.6 ms | stay on A
    747    48% |       88.8 ms |       75.0 ms | reroute to B
    530    57% |       91.6 ms |       58.9 ms | reroute to B
    558    77% |      126.5 ms |       65.4 ms | reroute to B
    730    94% |      169.1 ms |       80.8 ms | reroute to B

Rerouted 8 of 10 scenarios.
Average predicted time saved by rerouting: 54.8 ms

What this shows: the model learned that latency explodes when